# Phase 4 – Mini Project: Business Pipeline & Analytics
Built for the `sales.csv` + `customers.csv` sample datasets in Google Colab.

Covers: setup, cleaning, Tasks 1–7 (SQL + PySpark side by side), and saving output.

## Step 1: Install PySpark

In [ ]:
!pip install pyspark


## Step 2: Start Spark

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("BusinessPipeline").getOrCreate()
spark


## Step 3: Upload the datasets
Select `sales.csv` and `customers.csv` (or `customers__1_.csv`) when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()


## Step 4: Read the raw files
Adjust the filenames below if your uploaded file names differ (e.g. `customers__1_.csv`).

In [ ]:
from pyspark.sql.functions import (
    col, sum as _sum, count, when, to_date, concat_ws
)

sales = (spark.read.option("header", "true").option("inferSchema", "true")
    .csv("/content/sales.csv"))

customers = (spark.read.option("header", "true").option("inferSchema", "true")
    .csv("/content/customers.csv"))

sales.printSchema()
customers.printSchema()
sales.show(5)
customers.show(5)


## Step 5: Register raw temp views (needed for the SQL cells)

In [ ]:
sales.createOrReplaceTempView("sales_raw")
customers.createOrReplaceTempView("customers_raw")


## Data Cleaning
- Remove rows with null `customer_id`
- Remove duplicate rows
- Filter invalid values (`total_amount` <= 0)
- Parse `sale_date` correctly — the CSV stores dates as `dd-MM-yyyy` (e.g. `15-01-2024`), which Spark's default date inference will NOT parse correctly, so we convert it explicitly.

### PySpark

In [ ]:
sales_clean = (sales
    .dropna(subset=["customer_id"])
    .dropDuplicates()
    .filter(col("total_amount") > 0)
    .withColumn("sale_date", to_date(col("sale_date"), "dd-MM-yyyy")))

customers_clean = (customers
    .dropna(subset=["customer_id"])
    .dropDuplicates())

sales_clean.createOrReplaceTempView("sales")
customers_clean.createOrReplaceTempView("customers")

sales_clean.show(5)
customers_clean.show(5)


### SQL

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW sales AS
SELECT DISTINCT sale_id, customer_id, product_id,
       TO_DATE(sale_date, 'dd-MM-yyyy') AS sale_date,
       quantity, total_amount
FROM sales_raw
WHERE customer_id IS NOT NULL
  AND total_amount > 0
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT DISTINCT * FROM customers_raw
WHERE customer_id IS NOT NULL
""")

spark.sql("SELECT * FROM sales LIMIT 5").show()
spark.sql("SELECT * FROM customers LIMIT 5").show()


## Task 1: Daily Sales → `date, total_sales`

### PySpark

In [ ]:
task1 = (sales_clean
    .groupBy("sale_date")
    .agg(_sum("total_amount").alias("total_sales"))
    .orderBy("sale_date"))
task1.show(20)


### SQL

In [ ]:
task1_sql = spark.sql("""
SELECT sale_date AS date, SUM(total_amount) AS total_sales
FROM sales
GROUP BY sale_date
ORDER BY sale_date
""")
task1_sql.show(20)


## Task 2: City-wise Revenue → `city, total_revenue`

### PySpark

In [ ]:
task2 = (sales_clean.join(customers_clean, "customer_id")
    .groupBy("city")
    .agg(_sum("total_amount").alias("total_revenue"))
    .orderBy(col("total_revenue").desc()))
task2.show()


### SQL

In [ ]:
task2_sql = spark.sql("""
SELECT c.city, SUM(s.total_amount) AS total_revenue
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
GROUP BY c.city
ORDER BY total_revenue DESC
""")
task2_sql.show()


## Task 3: Top 5 Customers → `customer_name, total_spend`

### PySpark

In [ ]:
task3 = (sales_clean.join(customers_clean, "customer_id")
    .withColumn("customer_name", concat_ws(" ", col("first_name"), col("last_name")))
    .groupBy("customer_name")
    .agg(_sum("total_amount").alias("total_spend"))
    .orderBy(col("total_spend").desc())
    .limit(5))
task3.show()


### SQL

In [ ]:
task3_sql = spark.sql("""
SELECT CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
       SUM(s.total_amount) AS total_spend
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
GROUP BY CONCAT(c.first_name, ' ', c.last_name)
ORDER BY total_spend DESC
LIMIT 5
""")
task3_sql.show()


## Task 4: Repeat Customers (>1 order) → `customer_id, order_count`

### PySpark

In [ ]:
task4 = (sales_clean
    .groupBy("customer_id")
    .agg(count("sale_id").alias("order_count"))
    .filter(col("order_count") > 1))
task4.show()


### SQL

In [ ]:
task4_sql = spark.sql("""
SELECT customer_id, COUNT(sale_id) AS order_count
FROM sales
GROUP BY customer_id
HAVING COUNT(sale_id) > 1
""")
task4_sql.show()


## Task 5: Customer Segmentation → `customer_name, total_spend, segment`
Business rule: `total_spend > 10000` → Gold, `5000–10000` → Silver, else Bronze.

**Note:** this sample dataset's spend per customer is well under $5,000, so most/all customers will land in Bronze. That's expected given these fixed thresholds — not a bug.

### PySpark

In [ ]:
customer_spend = (sales_clean.join(customers_clean, "customer_id")
    .withColumn("customer_name", concat_ws(" ", col("first_name"), col("last_name")))
    .groupBy("customer_name")
    .agg(_sum("total_amount").alias("total_spend")))

task5 = customer_spend.withColumn(
    "segment",
    when(col("total_spend") > 10000, "Gold")
    .when((col("total_spend") >= 5000) & (col("total_spend") <= 10000), "Silver")
    .otherwise("Bronze")
).orderBy(col("total_spend").desc())

task5.show()


### SQL

In [ ]:
task5_sql = spark.sql("""
SELECT CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
       SUM(s.total_amount) AS total_spend,
       CASE
           WHEN SUM(s.total_amount) > 10000 THEN 'Gold'
           WHEN SUM(s.total_amount) BETWEEN 5000 AND 10000 THEN 'Silver'
           ELSE 'Bronze'
       END AS segment
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
GROUP BY CONCAT(c.first_name, ' ', c.last_name)
ORDER BY total_spend DESC
""")
task5_sql.show()


## Task 6: Final Reporting Table
Output: `customer_name, city, total_spend, order_count, segment`

### PySpark

In [ ]:
final_df = (sales_clean.join(customers_clean, "customer_id")
    .withColumn("customer_name", concat_ws(" ", col("first_name"), col("last_name")))
    .groupBy("customer_name", "city")
    .agg(
        _sum("total_amount").alias("total_spend"),
        count("sale_id").alias("order_count")
    )
    .withColumn(
        "segment",
        when(col("total_spend") > 10000, "Gold")
        .when((col("total_spend") >= 5000) & (col("total_spend") <= 10000), "Silver")
        .otherwise("Bronze")
    )
    .orderBy(col("total_spend").desc()))

final_df.show(50, truncate=False)


### SQL

In [ ]:
final_df_sql = spark.sql("""
SELECT CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
       c.city,
       SUM(s.total_amount) AS total_spend,
       COUNT(s.sale_id) AS order_count,
       CASE
           WHEN SUM(s.total_amount) > 10000 THEN 'Gold'
           WHEN SUM(s.total_amount) BETWEEN 5000 AND 10000 THEN 'Silver'
           ELSE 'Bronze'
       END AS segment
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
GROUP BY CONCAT(c.first_name, ' ', c.last_name), c.city
ORDER BY total_spend DESC
""")
final_df_sql.show(50, truncate=False)


## Task 7: Save Output
The assignment's path `/samples/output/report` is a Spark Playground path and doesn't exist in Colab — we use `/content/output/report` instead.

In [ ]:
final_df.write.mode("overwrite").csv("/content/output/report")
final_df_sql.write.mode("overwrite").csv("/content/output/report_sql")


### Optional: download the output folder as a zip

In [ ]:
!zip -r report.zip /content/output/report
from google.colab import files
files.download("report.zip")


## Reflection Questions

**Why is cleaning done before joining tables?**
Bad rows spread once you join. A null `customer_id` or a negative `total_amount` in `sales` will either silently drop out of an inner join or corrupt a sum after the join — cleaning first means the join only ever combines valid rows.

**What would go wrong if null keys are not removed?**
Rows with a null `customer_id` won't match anything in `customers` during the join (inner join drops them, left join keeps them with all-null customer fields) — either way you lose or misrepresent real sales, and downstream aggregates like total revenue quietly undercount.

**How did you decide join order?**
`sales` joined to `customers` on `customer_id`, since `sales` is the transactional (fact) table and `customers` is the lookup (dimension) table — you join fact to dimension so you get one row per transaction enriched with customer details, not one row per customer.

**Which step was most difficult and why?**
Usually the date parsing (`dd-MM-yyyy` in this dataset) and the segmentation thresholds — both run without errors but produce a wrong or unhelpful result if you don't check the actual data first.

**How is SQL logic similar to PySpark?**
Every SQL clause has a direct DataFrame method: `FROM`/`JOIN` → `.join()`, `WHERE` → `.filter()`, `GROUP BY` → `.groupBy()`, `HAVING` → a `.filter()` placed after `.agg()`, `CASE WHEN` → `when()`. Both compile down to the same underlying Spark execution plan.

**What challenges will appear with large data?**
Joins get expensive (shuffles across the cluster), `.show()`/`.collect()` become dangerous if used carelessly, and small mistakes like an unfiltered null key or duplicate row get amplified into much bigger data-quality problems at scale.

**Can you explain your pipeline in simple steps?**
Read the two raw files → clean nulls, duplicates, and invalid values → join sales to customers → aggregate into daily, city, and customer-level views → tag each customer with a Gold/Silver/Bronze segment → combine everything into one final reporting table → save it to disk.